# Multi-Subject Model Fitting (v3)

Fits the Bayesian listener model using `fit_listener` from the `bayesian_listener` package.

**New approach (v3):**
1. Estimates motor noise (`sigma_motor`) from lateral errors using ITD+ILD cues only
2. Fixes `sigma_motor` at the estimated value and `sigma_ild` at 1.0 dB
3. Fits only `sigma_spectral` and `sigma_prior` parameters

This methodology follows the parameter recovery validation approach and provides
better parameter identifiability by reducing the number of free parameters.

**Output:** `results/fitting_results_<timestamp>.csv`. The file used in the paper is
`results/fitting_results.csv` (notebook 08 later adds the `nll_500` column).
It feeds notebook 08, notebook 09, `stats/fitting_analysis.Rmd` (Friedman tests, ΔBIC) and the
figure scripts (Table 1, Table 2, Fig. 5, Fig. 6).

Refitting all 132 models takes several hours (≈7 h on 8 cores) and needs the participant HRTFs
and the behavioural data. By default (`RUN_FITTING = False`) the cached results are loaded instead.
The (trimmed) logs of the fitting cells are from the original run, made with a development version
of `bayesian_listener` preceding 0.2.0.

In [1]:
import numpy as np
import pandas as pd
import pyfar as pf
from pathlib import Path
import time
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings('ignore')

from bayesian_listener import BayesianListener
from bayesian_listener.fitting import fit_listener

from scipy import stats
from datetime import datetime

import paths

print('Imports complete')

Imports complete


## Load Data

In [2]:
TARGET_GRID = np.array([
    [315,  30], [315, -30], [180, -30], [ 90,  30],
    [120,   0], [ 90,   0], [225,  30], [330,   0],
    [270, -30], [  0, -30], [180,  30], [135,  30],
    [270,  30], [300,   0], [ 30,   0], [240,   0],
    [ 60,   0], [225, -30], [135, -30], [  0,  90],
    [210,   0], [270,  60], [  0,   0], [  0,  30],
    [ 90,  60], [180,  60], [180,   0], [270,   0],
    [ 45,  30], [ 45, -30], [  0,  60], [150,   0],
    [ 90, -30],
])

_tg_arr = np.column_stack([
    np.deg2rad(TARGET_GRID[:, 0]),
    np.deg2rad(TARGET_GRID[:, 1]),
    np.ones(len(TARGET_GRID)),
])
target_coords = pf.Coordinates.from_spherical_elevation(_tg_arr[:, 0], _tg_arr[:, 1], _tg_arr[:, 2])

print(f'{len(TARGET_GRID)} targets, {len(TARGET_GRID)} trials per simulation')

33 targets, 33 trials per simulation


In [3]:
# Behavioral responses
obs_tbl = pd.read_csv(paths.BEHAV_FILE)
obs_tbl = obs_tbl[(obs_tbl['condition'] == 'Measured')]
obs_tbl = obs_tbl[(obs_tbl['participant'] != 'P0247')]
trial_counts = obs_tbl.groupby('participant').size()

# first_14_participants = obs_tbl['participant'].unique()[:14]
# obs_tbl = obs_tbl[obs_tbl['participant'].isin(first_14_participants)]
# print(f"Using {len(first_14_participants)} participants: {list(first_14_participants)}")
subjects = obs_tbl['participant'].unique()

#print(trial_counts)
print(f'Subjects: {len(subjects)}')

Subjects: 33


## Configuration

In [4]:
RUN_FITTING = False  # True: refit all participants (hours); False: load the cached results

methods = ['SHMAX', 'SH', 'barycentric', 'barumerli2023']
num_repetitions = 200

# methods = ['SH']
# num_repetitions = 5

num_grid_points = 1
n_jobs = 14

print(f'Methods: {methods}')
print(f'MC repetitions: {num_repetitions}')
print(f'Grid points per dim: {num_grid_points}')
print(f'Parallel jobs: {n_jobs}')
print(f'Total fits: {len(subjects) * len(methods)}')

Methods: ['SHMAX', 'SH', 'barycentric', 'barumerli2023']
MC repetitions: 200
Grid points per dim: 1
Parallel jobs: 14
Total fits: 132


## Phase 1: Create Caches

In [5]:
if RUN_FITTING:
    print('Creating caches...')
    cache_start = time.time()

    for subj in subjects:
        sofa_path = str(paths.participant_sofa(subj))
        print(f'{subj}: ', end='', flush=True)
        for method in methods:
            model = BayesianListener(sofa_path)
            model.compute_template(interpolation=method)  # cached in data/preprocessed/
            print(f'{method} ', end='', flush=True)
        print()

    cache_time = time.time() - cache_start
    print(f'\nCache creation: {cache_time:.1f}s')
else:
    print('RUN_FITTING = False: skipping cache creation')

Creating caches...
P0001: ✓ Loading from cache: P0001_FreeFieldCompMinPhase_48kHz_5b84d57d_SHMAX_20260130_174300.pkl
✓ Cache loaded successfully
SHMAX ✓ Loading from cache: P0001_FreeFieldCompMinPhase_48kHz_5b84d57d_SH_20260130_154943.pkl
✓ Cache loaded successfully
SH ✓ Loading from cache: P0001_FreeFieldCompMinPhase_48kHz_5b84d57d_barycentric_20260130_174328.pkl
✓ Cache loaded successfully
barycentric ✓ Loading from cache: P0001_FreeFieldCompMinPhase_48kHz_5b84d57d_barumerli2023_20260130_174333.pkl
[... 292 lines trimmed ...]


## Phase 2: Fit All Subjects

In [6]:
if RUN_FITTING:
    print('='*60)
    print('Fitting all subjects in parallel')
    print('='*60)

    start_time = time.time()

    all_results = Parallel(n_jobs=-1, verbose=10)(
        delayed(fit_listener)(
            sofa_path=str(paths.participant_sofa(subj)),
            obs_tbl=obs_tbl,
            targets_coords=target_coords,
            interpolation_method=method,
            subject_id=subj,
            num_repetitions=num_repetitions,
            num_grid_points=num_grid_points,
            verbose=False
        )
        for subj in subjects
        for method in methods
    )

    fit_time = time.time() - start_time

    print(f'\nFitting: {fit_time/60:.1f} min')
    print(f'Total: {(cache_time + fit_time)/60:.1f} min')
    print(f'Average per fit: {fit_time/len(all_results):.1f}s')

    # Print summary
    for subj in subjects:
        subj_results = [r for r in all_results if r['subject'] == subj and r.get('success')]
        if subj_results:
            print(f'{subj}:')
            for r in subj_results:
                motor_info = f"(κ={r.get('kappa_motor', 'N/A'):.1f})" if 'kappa_motor' in r else ""
                print(f"  {r['method']:15s}: NLL={r['nll']:7.2f}  σ_motor={r['sigma_motor']:.2f}° {motor_info}")
        failed = [r for r in all_results if r['subject'] == subj and not r.get('success')]
        for r in failed:
            print(f"  {r['method']:15s}: FAILED - {r.get('error', '')}")
else:
    print('RUN_FITTING = False: skipping fitting')

[... original run: BADS optimisation logs and cache messages trimmed ...]
Fitting all subjects in parallel
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed: 18.9min
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed: 38.0min
[Parallel(n_jobs=-1)]: Done  16 tasks      | elapsed: 49.6min
[Parallel(n_jobs=-1)]: Done  25 tasks      | elapsed: 73.5min
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed: 100.7min
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed: 128.8min
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed: 158.9min
[Parallel(n_jobs=-1)]: Done  69 tasks      | elapsed: 200.4min
[Parallel(n_jobs=-1)]: Done  82 tasks      | elapsed: 237.9min
[Parallel(n_jobs=-1)]: Done  97 tasks      | elapsed: 280.9min
[Parallel(n_jobs=-1)]: Done 112 tasks      | elapsed: 330.5min
[Parallel(n_jobs=-1)]: Done 132 out of 132 | elapsed: 400.4min finished


## Results

In [ ]:
if RUN_FITTING:
    results_df = pd.DataFrame(all_results)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    results_file = paths.RESULTS_DIR / f'fitting_results_{timestamp}.csv'
    results_df.to_csv(results_file, index=False)
    print(f'Saved to: {results_file.relative_to(paths.ROOT)}')
else:
    results_file = paths.FITTING_RESULTS
    results_df = pd.read_csv(results_file).rename(columns={'participant': 'subject'})
    print(f'Loaded cached results: {results_file.relative_to(paths.ROOT)}')

In [ ]:
# Filter successful
successful_df = results_df[results_df['success'] == True].copy()
print(f'Successful: {len(successful_df)} / {len(results_df)}')

if len(successful_df) < len(results_df):
    failed = results_df[results_df['success'] == False]
    print('Failed fits:')
    print(failed[['subject', 'method', 'error']])

# Display
display_cols = ['subject', 'method', 'sigma_ild', 'sigma_spectral',
                'sigma_motor', 'kappa_motor', 'sigma_prior', 'nll']
print('\n' + '='*80)
print(successful_df[display_cols].to_string(index=False))

Successful: 132 / 132

subject        method  sigma_ild  sigma_spectral  sigma_motor  kappa_motor  sigma_prior        nll
  P0001         SHMAX        1.0        5.203930    11.957344    23.469666    37.468372 184.580694
  P0001            SH        1.0        4.457751    11.954450    23.480779    39.677346 183.020870
  P0001   barycentric        1.0        5.552319    11.989246    23.347693    36.413409 183.505604
  P0001 barumerli2023        1.0       10.000000    11.940041    23.536232    50.000000 265.876457
  P0006         SHMAX        1.0        8.313804     4.243595   182.797251    69.547344 812.154123
  P0006            SH        1.0       10.462930     4.221129   184.742903   179.899848 879.973759
  P0006   barycentric        1.0       13.171308     4.259302   181.455270   179.735584 828.961916
  P0006 barumerli2023        1.0       10.000000     4.332946   175.356453    50.000000 815.935983
  P0019         SHMAX        1.0        6.598694     8.689660    43.979907   179.88720

## Summary Statistics

In [ ]:
summary = successful_df.groupby('method')[
    ['sigma_ild', 'sigma_spectral', 'sigma_motor', 'sigma_prior', 'nll']
].agg(['mean', 'std'])
print('Summary Statistics by Method:')
print(summary.round(2))

Summary Statistics by Method:
              sigma_ild      sigma_spectral       sigma_motor        \
                   mean  std           mean   std        mean   std   
method                                                                
SH                  1.0  0.0          10.12  4.31       12.54  8.40   
SHMAX               1.0  0.0          10.35  5.25       12.52  8.40   
barumerli2023       1.0  0.0           9.53  4.06       12.53  8.22   
barycentric         1.0  0.0           9.50  3.23       12.51  8.41   

              sigma_prior            nll          
                     mean    std    mean     std  
method                                            
SH                  78.12  55.36  253.32  197.10  
SHMAX               68.96  51.76  249.84  185.11  
barumerli2023       55.03  35.65  251.36  185.44  
barycentric         73.49  57.38  249.66  193.13  
